### Forecasting HPC Workload Intensity with Vanilla LSTM

This notebook forecasts `num_jobs` from LLNL BlueGene/L SWF logs using a vanilla LSTM with fixed per-window z-score input normalization, trained with Adam. It is the non-DAIN baseline counterpart to `lstm-dain.ipynb`.

Set `FEATURE_MODE` to `'univariate'` or `'multivariate'` in the feature-selection cell to switch input features.

### Normalization note
There is no DAIN layer in this notebook.

**Inputs:** each multivariate lookback window is z-scored per feature using only that window's historical statistics (fixed / non-learned).

**Target:** `num_jobs` is normalized and inverted with a **raw per-window z-score** of the historical target channel only (Approach A), matching the LSTM-DAIN notebook's target space so the comparison isolates learned DAIN input normalization vs fixed window z-score.

## Step 1: Library Imports

In [ ]:
# Import necessary libraries
from sklearn.metrics import r2_score
from tabulate import tabulate
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.style.use("fivethirtyeight")
%matplotlib inline
from datetime import datetime


## Step 2: Data Preparation and Splitting

In [ ]:
import pandas as pd
import numpy as np

LABELS = (
    "job_id",
    "submit_time",
    "wait_time",
    "run_time",
    "allocated_proc",
    "cpu_time",
    "used_mem",
    "requested_proc",
    "requested_time",
    "requested_mem",
    "status",
    "user_id",
    "group_id",
    "executable_id",
    "queue_id",
    "partition_id",
    "preceding_job_id",
    "think_time",
)

def preprocess_data(window, data_directory):
    swf = pd.read_csv(
        data_directory,
        header=None,
        names=LABELS,
        index_col="job_id",
        delimiter=r"\s+",
        comment=";"
    )

    # Clean data from logs where run_time == 0
    swf = swf[swf['run_time'] > 0]

    swf['slice'] = swf.apply(lambda swf: swf['submit_time'] // (window*60), axis=1)

    swf.reset_index(inplace=True)

    swf['num_jobs'] = swf.groupby('slice')['slice'].transform('count')

    swf = swf.drop(["job_id"], axis=1)
    swf = swf[['slice', 'num_jobs', 'allocated_proc', 'run_time']]

    data_agg = swf.groupby(['slice', 'num_jobs']).agg(
        total_run_time=('run_time', 'sum'),
        std_run_time=('run_time', 'std'),
        total_allocated_proc=('allocated_proc', 'sum'),
        std_allocated_proc=('allocated_proc', 'std'),
    ).reset_index()

    data_agg['avg_allocated_proc'] = data_agg['total_allocated_proc'] / data_agg['num_jobs']
    data_agg['avg_run_time'] = data_agg['total_run_time'] / data_agg['num_jobs']

    cols = ['std_allocated_proc', 'std_run_time']
    data_agg[cols] = data_agg[cols].fillna(0)

    # 1. Define the full, uninterrupted range of expected slices
    min_slice = data_agg['slice'].min()
    max_slice = data_agg['slice'].max()
    all_slices = np.arange(min_slice, max_slice + 1)  # Assumes a step of 1.0

    # 2. Identify which slices are currently missing
    existing_slices = data_agg['slice'].unique()
    missing_slices = np.setdiff1d(all_slices, existing_slices)

    # 3. Create a new DataFrame specifically for the missing slices
    missing_data = pd.DataFrame({
        'slice': missing_slices,
        'num_jobs': 0,
        'total_allocated_proc': 0,
        'std_allocated_proc': 0,
        'total_run_time': 0,
        'std_run_time': 0,
        'avg_allocated_proc': 0,
        'avg_run_time': 0
    })

    data_agg = pd.concat([data_agg, missing_data], ignore_index=True)
    data_agg = data_agg.sort_values(by='slice').reset_index(drop=True)
    data_agg = data_agg.drop_duplicates(subset=['slice'])
    data_agg = data_agg.drop('slice', axis=1)

    return data_agg

df = preprocess_data(15, "/kaggle/input/datasets/salmanfaizhidayat/log-25-hpc/log-25/25-LLNL-uBGL-2006-2.swf")
df

In [ ]:
# Feature mode switch: 'univariate' uses only the target; 'multivariate' uses all listed covariates.
FEATURE_MODE = 'multivariate'  # 'univariate' | 'multivariate'

target_col = 'num_jobs'
MULTIVARIATE_FEATURE_COLS = [
    'num_jobs',
    'total_allocated_proc',
    'std_allocated_proc',
    'total_run_time',
    'std_run_time',
    'avg_allocated_proc',
    'avg_run_time',
]

if FEATURE_MODE == 'univariate':
    feature_cols = [target_col]
elif FEATURE_MODE == 'multivariate':
    feature_cols = list(MULTIVARIATE_FEATURE_COLS)
else:
    raise ValueError("FEATURE_MODE must be 'univariate' or 'multivariate'")

if target_col not in feature_cols:
    raise ValueError(f"target_col '{target_col}' must be included in feature_cols")

data = df[feature_cols].dropna()

target_idx = feature_cols.index(target_col)
n_features = len(feature_cols)

print("Feature mode:", FEATURE_MODE)
print("Features:", feature_cols)
print("Target:", target_col)
print("Number of features:", n_features)
print("Target index:", target_idx)

### Per-window z-score inside the forecasting model
Inputs are normalized with a fixed per-window z-score (all features) inside the LSTM model.

The future target uses a raw per-window z-score of the historical `num_jobs` channel only, then is inverted with those same raw statistics after prediction.

In [ ]:
# Vanilla LSTM helpers: per-window input z-score + raw target z-score (Approach A).
import tensorflow as tf
from tensorflow.keras.layers import Dense, LSTM, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

RAW_STD_FLOOR = 1e-3

def window_zscore(x, eps=RAW_STD_FLOOR):
    """Per-sample, per-feature z-score over the time axis of a window batch."""
    mean = tf.reduce_mean(x, axis=1, keepdims=True)
    std = tf.sqrt(tf.reduce_mean(tf.square(x - mean), axis=1, keepdims=True) + 1e-8)
    std = tf.maximum(std, eps)
    return (x - mean) / std

def raw_target_stats(x, target_idx, eps=RAW_STD_FLOOR):
    """Per-sample raw mean/std of the historical target channel."""
    target_window = x[:, :, target_idx:target_idx + 1]
    raw_avg = tf.reduce_mean(target_window, axis=1)
    raw_std = tf.sqrt(
        tf.reduce_mean(tf.square(target_window - tf.expand_dims(raw_avg, 1)), axis=1) + 1e-8
    )
    raw_std = tf.maximum(raw_std, eps)
    return raw_avg, raw_std

def normalize_target_raw(y, raw_avg, raw_std):
    return (y - raw_avg) / raw_std

def inverse_target_raw(normalized, raw_avg, raw_std):
    return normalized * raw_std + raw_avg

def raw_target_stats_numpy(x, target_idx, eps=RAW_STD_FLOOR):
    target_window = x[:, :, target_idx:target_idx + 1]
    raw_avg = np.mean(target_window, axis=1)
    raw_std = np.sqrt(np.mean(np.square(target_window - np.expand_dims(raw_avg, 1)), axis=1) + 1e-8)
    raw_std = np.maximum(raw_std, eps)
    return raw_avg, raw_std

def inverse_target_raw_numpy(normalized, raw_avg, raw_std):
    return normalized * raw_std + raw_avg


In [ ]:
class ForecastModel(tf.keras.Model):
    """Vanilla LSTM forecast model with fixed per-window input z-score."""
    def __init__(self, backbone_layers, target_idx, **kwargs):
        super().__init__(**kwargs)
        self.backbone_layers = backbone_layers
        self.target_idx = target_idx
        self.loss_tracker = tf.keras.metrics.Mean(name='loss')

    def call(self, inputs, training=False):
        x = window_zscore(inputs)
        for layer in self.backbone_layers:
            x = layer(x, training=training)
        return x

    def compile(self, optimizer, loss='mean_squared_error', **kwargs):
        super().compile(optimizer=optimizer, **kwargs)
        self.loss_fn = tf.keras.losses.get(loss)

    @property
    def metrics(self):
        return [self.loss_tracker]

    def train_step(self, data):
        x, y = data
        with tf.GradientTape() as tape:
            pred = self(x, training=True)
            raw_avg, raw_std = raw_target_stats(x, self.target_idx)
            y_norm = normalize_target_raw(y, raw_avg, raw_std)
            loss = self.loss_fn(y_norm, pred)
            if self.losses:
                loss += tf.add_n(self.losses)

        grads = tape.gradient(loss, self.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.trainable_variables))
        self.loss_tracker.update_state(loss)
        return {'loss': self.loss_tracker.result()}

    def test_step(self, data):
        x, y = data
        pred = self(x, training=False)
        raw_avg, raw_std = raw_target_stats(x, self.target_idx)
        y_norm = normalize_target_raw(y, raw_avg, raw_std)
        loss = self.loss_fn(y_norm, pred)
        if self.losses:
            loss += tf.add_n(self.losses)
        self.loss_tracker.update_state(loss)
        return {'loss': self.loss_tracker.result()}


training_callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6),
]

In [ ]:
# Convert the selected feature frame to a numpy array (respects FEATURE_MODE / feature_cols)
dataset = data.values.astype(np.float32)

# Split the dataset into training, validation, and test sets
training_data_len = int(np.ceil(len(dataset) * 0.70))
validation_data_len = int(np.ceil(len(dataset) * 0.85))

train_data = dataset[:training_data_len, :]
val_data = dataset[training_data_len:validation_data_len, :]
test_data = dataset[validation_data_len:, :]

print(f"Training Data Length: {len(train_data)}")
print(f"Validation Data Length: {len(val_data)}")
print(f"Test Data Length: {len(test_data)}")

### Data scaling / normalization
No global scaler is fitted. The raw series is kept intact. Inside the model, each input window is z-scored per feature; the target uses raw per-window z-score statistics from historical `num_jobs` only.

In [ ]:
# Keep the raw series intact. Per-window z-score is applied inside the model on each window.
full_data = dataset.astype(np.float32)
train_end = len(train_data)
val_end = train_end + len(val_data)
print('Configured for per-window input z-score; target uses raw z-score.')
print(f"Train boundary: {train_end} | Validation boundary: {val_end} | Total: {len(full_data)}")

## Sequence Construction and Sliding Window Tuning
Sequences are built from the raw multivariate series. Lookback may use prior-split history so validation/test targets start at each split boundary. Per-window z-score is applied inside the vanilla LSTM model.

In [ ]:
# Vanilla LSTM model builder used for both window tuning and final training.
def build_model(optimizer, window_size, n_features, target_idx):
    backbone_layers = [
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1),
    ]

    model = ForecastModel(
        backbone_layers=backbone_layers,
        target_idx=target_idx,
        name='Vanilla_LSTM'
    )
    model.compile(optimizer=optimizer, loss='mean_squared_error')

    _ = model(tf.zeros((1, window_size, n_features)), training=False)
    return model

print('Vanilla LSTM model builder defined.')

### Sliding Window Hyperparameter Tuning
We search over candidate lookback windows using vanilla LSTM + Adam on validation Iverson RMSE (original `num_jobs` scale, raw inverse; scored only where actual ≠ 0), then lock in the best window for the final training run.


In [ ]:
def create_sequences(data, window_size, start_idx, end_idx, target_idx):
    """Build sequences where start_idx is the first target index.

    Lookback may draw history from before start_idx (prior split), avoiding a
    window-sized gap at train/val/test boundaries.
    """
    x, y = [], []

    for i in range(start_idx, end_idx):
        if i < window_size:
            continue
        x.append(data[i - window_size:i, :])
        y.append(data[i, target_idx])

    x = np.array(x, dtype=np.float32)
    y = np.array(y, dtype=np.float32).reshape(-1, 1)
    return x, y


def make_predictions(model, x_data, target_idx):
    """Invert model outputs with the same raw per-window stats used in training."""
    normalized_pred = model.predict(x_data, verbose=0)
    raw_avg, raw_std = raw_target_stats_numpy(x_data, target_idx)
    return inverse_target_raw_numpy(normalized_pred, raw_avg, raw_std)


def _flatten(array):
    return np.asarray(array, dtype=float).reshape(-1)


def iverson_mse(actual, prediction):
    """MSE with Iverson bracket: score only where actual != 0."""
    actual, prediction = _flatten(actual), _flatten(prediction)
    mask = actual != 0.0
    n = int(mask.sum())
    if n == 0:
        return np.nan
    return float(((prediction - actual) ** 2 * mask).sum() / n)


def iverson_mae(actual, prediction):
    """MAE with Iverson bracket: score only where actual != 0."""
    actual, prediction = _flatten(actual), _flatten(prediction)
    mask = actual != 0.0
    n = int(mask.sum())
    if n == 0:
        return np.nan
    return float((np.abs(prediction - actual) * mask).sum() / n)


def iverson_rmse(actual, prediction):
    mse = iverson_mse(actual, prediction)
    return float(np.sqrt(mse)) if np.isfinite(mse) else np.nan


def iverson_r2(actual, prediction):
    """R^2 with Iverson bracket: fit score only where actual != 0."""
    actual, prediction = _flatten(actual), _flatten(prediction)
    mask = actual != 0.0
    if int(mask.sum()) < 2:
        return np.nan
    return float(r2_score(actual[mask], prediction[mask]))


def iverson_rmse_in_original_scale(model, x_data, y_true, target_idx):
    y_pred = make_predictions(model, x_data, target_idx)
    return iverson_rmse(y_true, y_pred)


In [ ]:
WINDOW_CANDIDATES = [30, 60, 90, 120]

### Vanilla LSTM Adam Window Tuning

In [ ]:
window_tuning_results = []
best_window_size = WINDOW_CANDIDATES[0]
best_window_val_rmse = float('inf')

In [ ]:
print('Tuning sliding window on validation Iverson RMSE (raw inverse)...')
for candidate_window in WINDOW_CANDIDATES:
    x_tune_train, y_tune_train = create_sequences(full_data, candidate_window, 0, train_end, target_idx)
    x_tune_val, y_tune_val = create_sequences(full_data, candidate_window, train_end, val_end, target_idx)

    if len(x_tune_val) == 0:
        print(f'  window={candidate_window:3d} -> skipped (0 validation samples)')
        continue

    tune_model = build_model(
        Adam(learning_rate=0.001), candidate_window, n_features, target_idx
    )
    tune_model.fit(
        x_tune_train, y_tune_train,
        validation_data=(x_tune_val, y_tune_val),
        batch_size=32,
        epochs=50,
        verbose=0,
        callbacks=training_callbacks,
    )

    candidate_val_rmse = iverson_rmse_in_original_scale(tune_model, x_tune_val, y_tune_val, target_idx)
    window_tuning_results.append((candidate_window, candidate_val_rmse))
    print(f'  window={candidate_window:3d} -> validation I-RMSE: {candidate_val_rmse:.4f}')

    if candidate_val_rmse < best_window_val_rmse:
        best_window_val_rmse = candidate_val_rmse
        best_window_size = candidate_window

print(f'\nSelected sliding window: {best_window_size} (validation I-RMSE: {best_window_val_rmse:.4f})')

x_train, y_train = create_sequences(full_data, best_window_size, 0, train_end, target_idx)
x_val, y_val = create_sequences(full_data, best_window_size, train_end, val_end, target_idx)
x_test, y_test = create_sequences(full_data, best_window_size, val_end, len(full_data), target_idx)

assert len(x_val) > 0, f'Validation set empty for window={best_window_size}'
assert len(x_test) > 0, f'Test set empty for window={best_window_size}'

print('Data preparation completed.')
print(f'Train samples: {len(x_train)}, Val samples: {len(x_val)}, Test samples: {len(x_test)}')


In [ ]:
# Visualize sliding window tuning results
window_df = pd.DataFrame(window_tuning_results, columns=['Window Size', 'Validation I-RMSE'])
plt.figure(figsize=(8, 4))
plt.bar(window_df['Window Size'].astype(str), window_df['Validation I-RMSE'], color='steelblue')
plt.axhline(best_window_val_rmse, color='crimson', linestyle='--', label=f'Best: {best_window_size}')
plt.title('Sliding Window Hyperparameter Tuning — Vanilla LSTM Adam (Iverson)')
plt.xlabel('Lookback Window (15-min slices)')
plt.ylabel('Validation I-RMSE (num_jobs, actual ≠ 0)')
plt.legend()
plt.tight_layout()
plt.show()

## Step 3: Training Vanilla LSTM and Evaluating Performance

In [ ]:
BATCH_SIZE = 32
EPOCHS = 50

**Training Vanilla LSTM with Adam**

In [ ]:
# Vanilla LSTM with Adam
optimizer_adam = Adam(learning_rate=0.001)
model_lstm = build_model(optimizer_adam, best_window_size, n_features, target_idx)
history_lstm = model_lstm.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=training_callbacks,
    verbose=1
)
print('Vanilla LSTM with Adam training completed.')

## Step 4: Making Predictions and Evaluating the Model

In [ ]:
# Predictions are inverted with the same raw per-window stats used during training.
# Metrics use Iverson's bracket: only time slices where actual num_jobs != 0 are scored.
y_test_actual = y_test
pred_lstm = make_predictions(model_lstm, x_test, target_idx)

metrics_values = {
    'Vanilla LSTM Adam': {
        'I-MSE': iverson_mse(y_test_actual, pred_lstm),
        'I-RMSE': iverson_rmse(y_test_actual, pred_lstm),
        'I-MAE': iverson_mae(y_test_actual, pred_lstm),
        'I-R2': iverson_r2(y_test_actual, pred_lstm),
    }
}

print('Predictions and Iverson metric calculations completed.')
for model_name, m in metrics_values.items():
    print(
        f"  {model_name}: I-MSE = {m['I-MSE']:.4f} | I-RMSE = {m['I-RMSE']:.4f} | "
        f"I-MAE = {m['I-MAE']:.4f} | I-R^2 = {m['I-R2']:.4f}"
    )


In [ ]:
# Ground truth vs prediction
plt.figure(figsize=(16, 5))
plt.plot(y_test_actual, label='Ground Truth', linewidth=1.5)
plt.plot(pred_lstm, label='Prediction', linewidth=1.5, alpha=0.85)
plt.title('Vanilla LSTM Adam — Ground Truth vs Prediction (Test Set)')
plt.xlabel('Test Sample')
plt.ylabel('num_jobs')
plt.legend()
plt.tight_layout()
plt.show()

## Step 5: Visualizations and Analysis
Visualizations included are:
* Line Plot of Loss vs. Epochs
* Line Plot of Log(Loss) vs. Epochs
* Line Plot of Validation Loss vs. Epochs
* Bar chart of Final loss
* Iverson metrics summary table

**Line Plot of Loss vs. Epochs**

In [ ]:
# Line Plot of Loss vs. Epochs
plt.figure(figsize=(14, 7))
plt.plot(history_lstm.history['loss'], label='Vanilla LSTM Adam')
plt.title('Loss vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

print("Loss vs. Epochs plot generated.")

In [ ]:
# Convergence Curves
plt.figure(figsize=(14, 7))
plt.plot(np.log10(history_lstm.history['loss']), label='Vanilla LSTM Adam')
plt.title('Log(Loss) vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Log(Loss)')
plt.legend()
plt.show()

print("Convergence curves generated.")

In [ ]:
# Line Plot of Validation Loss vs. Epochs
plt.figure(figsize=(14, 7))
plt.plot(history_lstm.history['val_loss'], label='Vanilla LSTM Adam')
plt.title('Validation Loss vs. Epochs')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.legend()
plt.show()

print("Validation Loss vs. Epochs plot generated.")

**Bar Chart of Final Loss**

In [ ]:
# Bar Chart of Final Loss
final_losses = [history_lstm.history['loss'][-1]]
models = ['Vanilla LSTM Adam']

plt.figure(figsize=(6, 5))
plt.bar(models, final_losses, color='steelblue')
plt.title('Final Loss — Vanilla LSTM Adam')
plt.xlabel('Model')
plt.ylabel('Final Loss')
plt.show()

print("Final Loss bar chart generated.")

**Metrics Summary Table**

In [ ]:
# Metrics Summary Table (Iverson brackets: actual != 0)
metrics_table = [
    [
        model,
        f"{m['I-MSE']:.4f}",
        f"{m['I-RMSE']:.4f}",
        f"{m['I-MAE']:.4f}",
        f"{m['I-R2']:.4f}",
    ]
    for model, m in metrics_values.items()
]
print(tabulate(
    metrics_table,
    headers=['Model', 'I-MSE', 'I-RMSE', 'I-MAE', 'I-R^2'],
    tablefmt='pretty'
))

print("Iverson metrics summary table generated.")
